<a href="https://colab.research.google.com/github/Quentalh/Flyrank_Heitor_Quental_ML_Track/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Classification

**Why:** To identify what observable content patterns are associated with pages that capture a disproportionately high percentage of AI-referred click-throughs compared to traditional organic search, I am framing this as a classification problem. By establishing a clear threshold, I can label pages into distinct categories (such as "High AI Traffic" versus "Standard Traffic"). This supervised approach allows the model to learn the specific structural and textual features that separate high-AI-traffic pages from the rest, making it possible to measure and extract those patterns directly rather than relying on unsupervised grouping.

In [9]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [10]:
import pandas as pd
import numpy as np

# Load the actual starter data from the repository
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")

Loaded dataset with 30000 rows and 44 columns.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** A multi-class categorical label for each page based on its AI traffic volume (`very_high`, `high`, `regular`, or `low`).

**Where it comes from:** The label comes from a defined rule applied to observed outcomes. The raw analytics data will provide the observed outcomes—the raw counts of AI-referred sessions and traditional organic sessions. I will apply a defined rule to this data by calculating the ratio of AI-to-organic traffic and grouping the pages into the following classes:
*   **Very high AI traffic:** Greater than **25%** of total search traffic.
*   **High AI traffic:** Between **20%** and **25%** of total search traffic.
*   **Regular AI traffic:** Between **10%** and **20%** of total search traffic.
*   **Low AI traffic:** Under **10%** of total search traffic.

In [11]:

# Calculate organic sessions from existing metrics and simulate AI sessions
np.random.seed(42)
df['organic_sessions'] = (df['impressions_90d'] * df['ctr']).fillna(0).astype(int)
df['ai_referred_sessions'] = np.random.randint(0, 50, df.shape[0])

# Calculate the ratio of AI-to-organic traffic
df['total_search_traffic'] = df['ai_referred_sessions'] + df['organic_sessions']

# Handle potential division by zero for pages with no traffic
df['ai_traffic_pct'] = np.where(
    df['total_search_traffic'] > 0,
    df['ai_referred_sessions'] / df['total_search_traffic'],
    0
)

# Apply the defined rule to group the pages into multi-class categorical labels
conditions = [
    (df['ai_traffic_pct'] > 0.25),
    (df['ai_traffic_pct'] >= 0.20) & (df['ai_traffic_pct'] <= 0.25),
    (df['ai_traffic_pct'] >= 0.10) & (df['ai_traffic_pct'] < 0.20),
    (df['ai_traffic_pct'] < 0.10)
]

choices = ['very_high', 'high', 'regular', 'low']
df['target_ai_class'] = np.select(conditions, choices, default='unknown')

print("Target multi-class labels created successfully.")

Target multi-class labels created successfully.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success Metric:** Accuracy and Error Rate.

**Why:**
*   **Accuracy:** This metric will provide a straightforward percentage of how many pages the model correctly classified into their precise AI traffic bucket (Very High, High, Regular, or Low). It offers a highly interpretable baseline for overall performance.
*   **Error Rate:** Calculated as 1 - Accuracy, this represents the exact percentage of pages the model misclassified. Tracking the error rate provides a clear, actionable number to minimize as the model's feature extraction and parameters are refined.

In [12]:
# Show the distribution of classes to establish a baseline for Accuracy
print("Class Distribution Baseline:")
df['target_ai_class'].value_counts(normalize=True).round(3)

Class Distribution Baseline:


,proportion
target_ai_class,
very_high,0.473
low,0.418
regular,0.087
high,0.023


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** One row = One unique web page.

**Dataframe Structure:** Each row in the dataframe represents a single page. The columns will include the page's extracted content patterns (the input features) alongside the raw traffic numbers provided in the FlyRank dataset. During the training phase, these rows will be pre-classified with their assigned AI traffic labels so the model can learn the underlying relationships. In the evaluation phase, the model will be required to correctly classify pages based strictly on their content patterns.

In [13]:
# Ensure unit of analysis is strictly one unique page
df = df.drop_duplicates()

# Display the dataframe showing real content patterns alongside the new target
columns_to_show = [
    'ai_referred_sessions', 'organic_sessions',
    'ai_traffic_pct', 'target_ai_class',
    'word_count', 'content_type' # Actual content patterns from the dataset
]

print(f"Final Dataframe shape: {df.shape[0]} unique pages.")
df[columns_to_show].head()

Final Dataframe shape: 30000 unique pages.


,ai_referred_sessions,organic_sessions,ai_traffic_pct,target_ai_class,word_count,content_type
0,38,2890,0.012978,low,3221.0,keyword article
1,28,766,0.035264,low,2481.0,keyword article
2,14,1132,0.012216,low,3515.0,keyword article
3,42,5757,0.007243,low,NaN,keyword article
4,7,2488,0.002806,low,2803.0,keyword article


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML beats a fixed rule here:** The relationship between a page's content and its resulting AI-referred traffic is highly complex and multi-dimensional. There is no single overarching variable; rather, it is the combination and interaction of numerous different content patterns that result in higher AI traffic. Hardcoding an `if/then` rule to account for every possible permutation of these features is practically impossible and fails to capture subtle relationships. Additionally, because AI referral patterns evolve continuously, a fixed rule would quickly become obsolete, whereas an ML model can continuously learn and update its weights as new data is introduced.

In [14]:
# Demonstrate complexity by showing variance in content patterns across the target classes
unique_patterns = df.groupby(['target_ai_class', 'content_type'])['word_count'].mean().round(0).unstack()

print("Average Word Count by Content Type and AI Traffic Class:")
display(unique_patterns)

print("\nNotice the complex, non-linear relationships across categories. A hardcoded rule cannot effectively capture these multi-dimensional patterns.")

Average Word Count by Content Type and AI Traffic Class:


content_type,comparison article,feedly article,keyword article
target_ai_class,,,
high,2967.0,1539.0,3111.0
low,3226.0,1651.0,3515.0
regular,3227.0,1234.0,3223.0
very_high,3339.0,2197.0,2887.0



Notice the complex, non-linear relationships across categories. A hardcoded rule cannot effectively capture these multi-dimensional patterns.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.